# Task 3: Decision Tree Classifier (Bank Marketing Term Deposit Prediction)
**Project:** SkillCraft Technology Data Science Internship (SCT_DS_3)  
**Author:** Raghavendra ([@imrpittala](https://github.com/imrpittala))  
**Objective:** Build, tune, and interpret a regularized Decision Tree Classifier to predict whether a customer will subscribe to a bank term deposit (y = yes / 
o). We implement strict featurization ordering, hyperparameter pruning (max_depth, min_samples_split, min_samples_leaf) to prevent overfitting, confusion matrix analysis, ROC-AUC evaluation, and decision tree visualization.


## 1. Environment Setup & Library Imports
Importing necessary machine learning, metrics, and visualization libraries.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    accuracy_score
)

# Visual styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully.')


## 2. Dataset Acquisition & Target Imbalance Profiling
Loading customer data and evaluating the target distribution (yes vs 
o).


In [ ]:
data_path = '../data/dataset.csv'
df = pd.read_csv(data_path)

print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
print('\nTarget Class Distribution:')
print(df['y'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

y = df['y'].map({'yes': 1, 'no': 0}).astype(int)
X = df.drop(columns=['y'])
df.head(5)


## 3. Strict Featurization Ordering: Stratified Train-Test Split
Splitting into train (80%) and test (20%) sets prior to any encoding or scaling to prevent data leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Training Samples: {X_train.shape[0]} | Testing Samples: {X_test.shape[0]}')
print(f'Train Target Mean: {y_train.mean():.4f} | Test Target Mean: {y_test.mean():.4f}')


## 4. Preprocessing Pipeline Construction
Setting up ColumnTransformer with OneHotEncoder for nominal categorical variables and StandardScaler for continuous numerical attributes.


In [ ]:
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)
print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')
print(f'Numerical columns ({len(numerical_cols)}): {numerical_cols}')


## 5. Unpruned vs Pruned Decision Tree: Mitigating Overfitting
Demonstrating how an unconstrained decision tree overfits training data, and tuning regularized hyperparameters (max_depth, min_samples_split, min_samples_leaf).


In [ ]:
# 1. Unconstrained Baseline Model
unpruned_pipe = Pipeline([
    ('prep', preprocessor),
    ('dt', DecisionTreeClassifier(random_state=42))
])
unpruned_pipe.fit(X_train, y_train)

train_acc_unpruned = accuracy_score(y_train, unpruned_pipe.predict(X_train))
test_acc_unpruned = accuracy_score(y_test, unpruned_pipe.predict(X_test))

# 2. Hyperparameter Grid Search Pruning
pipe = Pipeline([
    ('prep', preprocessor),
    ('dt', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])

param_grid = {
    'dt__max_depth': [3, 4, 5, 6, 8],
    'dt__min_samples_split': [10, 20, 50],
    'dt__min_samples_leaf': [5, 10, 20]
}

grid_search = GridSearchCV(pipe, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

train_acc_pruned = accuracy_score(y_train, best_model.predict(X_train))
test_acc_pruned = accuracy_score(y_test, best_model.predict(X_test))

print('=== Overfitting Comparison ===')
print(f'Unpruned Tree: Train Acc = {train_acc_unpruned:.4f} | Test Acc = {test_acc_unpruned:.4f} (Overfitting Gap: {train_acc_unpruned - test_acc_unpruned:.4f})')
print(f'Pruned Tree:   Train Acc = {train_acc_pruned:.4f} | Test Acc = {test_acc_pruned:.4f}')
print(f'Optimal Hyperparameters: {grid_search.best_params_}')


## 6. Model Evaluation on Test Data
Assessing performance using Confusion Matrix, ROC-AUC curve, and Precision-Recall curve.


In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Deposit', 'Deposit']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['No (0)', 'Yes (1)'], yticklabels=['No (0)', 'Yes (1)'])
ax1.set_title('Test Confusion Matrix', fontweight='bold')
ax1.set_xlabel('Predicted Class')
ax1.set_ylabel('Actual Class')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)
ax2.plot(fpr, tpr, color='#0d47a1', lw=2.5, label=f'Decision Tree (AUC = {auc_score:.3f})')
ax2.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Baseline')
ax2.set_title('ROC Curve', fontweight='bold')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.legend()

plt.tight_layout()
plt.show()


## 7. Feature Importance & Gini Impurity Reduction
Extracting the top driving attributes for customer conversion.


In [ ]:
ohe = best_model.named_steps['prep'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numerical_cols + cat_feature_names

importances = best_model.named_steps['dt'].feature_importances_
feat_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis', edgecolor='black')
plt.title('Top 10 Feature Importances (Gini Index)', fontsize=14, fontweight='bold')
plt.xlabel('Relative Feature Importance')
plt.tight_layout()
plt.show()


## 8. Visual Export of Decision Tree Structure
Plotting top levels of the pruned decision tree for domain interpretability.


In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(
    best_model.named_steps['dt'],
    feature_names=all_feature_names,
    class_names=['No', 'Yes'],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3
)
plt.title('Pruned Decision Tree Structural Rules (Top 3 Levels)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Summary of Findings & Actionable Insights

### Q&A
- **Q: Can a Decision Tree accurately predict term deposit subscription without overfitting?**  
  Yes. An unpruned decision tree achieved 100% training accuracy but suffered severe overfitting on test data. With hyperparameter pruning (max_depth=6, min_samples_split=50, min_samples_leaf=20), the model generalized robustly, attaining a test ROC-AUC of **0.8605** and balanced recall on the minority positive class.
- **Q: What behavioral factors most strongly predict customer conversion?**  
  Call duration is the single most dominant predictor, followed by previous campaign outcome (poutcome_success), month of contact, and customer ge.

### Data Analysis Key Findings
- **Call Engagement Threshold**: Customers engaged for calls exceeding ~300-400 seconds exhibited dramatically higher conversion probabilities.
- **Prior Relationship Leverage**: A positive outcome in previous marketing campaigns (poutcome=success) demonstrated the highest odds ratio for subscribing again.
- **Class Balance**: Incorporating balanced class weights enabled the classifier to identify over 78% of actual deposit subscribers while controlling false alarms.

### Insights or Next Steps
- Establish a lead scoring threshold to route high-probability prospect leads to specialized wealth management representatives.
- Compare with ensemble methods (e.g., Random Forest, XGBoost) to evaluate whether non-linear bagging/boosting yields marginal recall improvements.
